# Does Civil Resistance Work? — Pilot exploration

**Research question:** When does civil resistance produce a policy concession, leadership change or institutional transition, and when do major demands remain unmet?

This notebook explores a **handpicked 12-case pilot dataset** centered on Albania. It is **not** a worldwide sample. Counts below are descriptive only; do not convert them into estimates of campaign success or causal effects. **Original cutoff: 2026-10-18; targeted Albania review: 2026-10-20.**

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

possible = [Path("data/cases.csv"), Path("../data/cases.csv")]
data_path = next((p for p in possible if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Run Jupyter from the project root or notebooks/ directory, or provide the CSV path.")
df = pd.read_csv(data_path)
print(f"Loaded {len(df)} selected cases from {data_path}")
display(df[["case_id", "country", "start_year", "status", "immediate_outcome"]])

## 1. Validate the research table

We check identifiers, missing core fields and whether an ongoing campaign was incorrectly assigned a final outcome. Passing these checks does **not** validate historical interpretations.

In [ ]:
required = ["case_id", "country", "campaign", "start_year", "end_year",
            "status", "campaign_style", "demand_scope", "immediate_outcome", "source_1"]
assert df["case_id"].is_unique, "Duplicate case IDs"
assert df[required].notna().all().all(), "Missing required values"
assert (df["start_year"] <= df["end_year"]).all(), "Invalid year ranges"
assert (df.loc[df["status"].eq("ongoing"), "immediate_outcome"] == "unresolved").all()
print("Structural checks passed.")
print("Null values by column:")
display(df.isna().sum().to_frame("null_count"))

## 2. Albania over time

1997 is a mixed/armed state crisis rather than a straightforward nonviolent campaign. The 2026 entry is **ongoing**; its endpoint is explicitly unknown and its campaign style is `uncertain`. See docs/ALBANIA_REVIEW_2026-10-20.md.

In [ ]:
albania = df.loc[df["country"].eq("Albania"),
                 ["case_id", "start_year", "end_year", "campaign",
                  "campaign_style", "status", "immediate_outcome", "observed_result"]]
display(albania.sort_values("start_year"))

## 3. Outcome categories in the *selected* sample

A frequency chart about these rows only. It is **not** a success-rate chart; outcome categories are not equivalent to the movements' original goals.

In [ ]:
counts = df["immediate_outcome"].value_counts().sort_values()
ax = counts.plot(kind="barh", figsize=(9, 5))
ax.set_title("Provisional outcome codes — selected pilot cases only")
ax.set_xlabel("Number of handpicked case records")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 4. Compare completed predominantly nonviolent cases

These labels are **provisional qualitative judgments**, not event-by-event violence measurements. The filtered list is still *not* statistically representative.

In [ ]:
subset = df.loc[
    df["status"].eq("completed") &
    df["campaign_style"].eq("predominantly_nonviolent"),
    ["case_id", "country", "campaign", "demand_scope", "immediate_outcome"]
]
display(subset.sort_values(["country", "case_id"]))
display(pd.crosstab(subset["demand_scope"], subset["immediate_outcome"]))

## 5. Different political results, similar protest headlines

Investigate Albania 2018 (education concessions), Hong Kong 2019 (one major bill withdrawn), Serbia 2000 (regime transition) and Belarus 2020 (core demands unmet). Their goals and political institutions differ, so this is **hypothesis generation**, not a controlled comparison.

In [ ]:
ids = ["ALB-2018", "HKG-2019", "SRB-2000", "BLR-2020"]
display(
    df.loc[df["case_id"].isin(ids),
           ["case_id", "demand_scope", "immediate_outcome",
            "comparison_question", "long_term_caution"]]
      .set_index("case_id").reindex(ids)
)

## 6. Research questions for the next dataset

- Does opposition success require defections or noncooperation from coercive institutions, or are they outcomes of mobilization?
- How should temporary concessions be distinguished from durable policy implementation?
- Which sources can provide **consistent** measures of coalition breadth, repression, socioeconomic conditions and elite cohesion?
- Are later democratic reversals better studied as a **separate outcome** from initial protest success?

**Next step:** obtain and read the NAVCO 2.1 codebook before any hypothesis testing. Do **not** append these case records to NAVCO directly.

See [the methodology](../docs/METHODOLOGY.md) and [source list](../docs/SOURCES.md).